<a href="https://colab.research.google.com/github/prathampharmaai/Pratham-/blob/main/RD_kit_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import rdkit
from rdkit.Chem import Descriptors, Crippen, AllChem as Chem
from rdkit import DataStructs
import pandas as pd

smiles = {'Aspirin':'CC(=O)Oc1ccccc1C(=O)O',
'Paracetamol':'CC(=O)Nc1ccc(O)cc1',
'Ibuprofen':'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
'Atenolol':'CC(C)NCC(O)COc1ccc(CC(N)=O)cc1',
'Metformin':'CN(C)C(=N)NC(N)=N'}

rows = []

for name, smil in smiles.items():
  mol = Chem.MolFromSmiles(smil)
  mw = Descriptors.MolWt(mol)
  logp = Crippen.MolLogP(mol)
  hbd = Chem.rdMolDescriptors.CalcNumHBD(mol)
  hba = Chem.rdMolDescriptors.CalcNumHBA(mol)
  rows.append([name,mw,logp, hbd, hba])

df = pd.DataFrame(rows,columns=['Name','MW','LogP','hbd','hba'])
df['lipinski'] = ((df['MW']<500) & (df['LogP']<5) & (df['hbd']<=5) & (df['hba']<=10))
df

,Name,MW,LogP,hbd,hba,lipinski
0,Aspirin,180.159,1.31010,1,3,True
1,Paracetamol,151.165,1.35060,2,2,True
2,Ibuprofen,206.285,3.07320,1,1,True
3,Atenolol,266.341,0.45210,3,4,True
4,Metformin,129.167,-1.03416,4,2,True


In [10]:
from rdkit.Chem import rdFingerprintGenerator

parac = Chem.MolFromSmiles(smiles['Paracetamol'])
aspirin = Chem.MolFromSmiles(smiles['Aspirin'])
ibuprofen = Chem.MolFromSmiles(smiles['Ibuprofen'])

mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
fp_par = mfgen.GetFingerprint(parac)
fp_asp = mfgen.GetFingerprint(aspirin)
fp_ibu = mfgen.GetFingerprint(ibuprofen)

sim_asp_par = DataStructs.TanimotoSimilarity(fp_par, fp_asp)
sim_ibu_par = DataStructs.TanimotoSimilarity(fp_par, fp_ibu)

print(f'Paracetamol vs Aspirin: {sim_asp_par:.3f}')
print(f'Paracetamol vs Ibuprofen: {sim_ibu_par:.3f}')

Paracetamol vs Aspirin: 0.229
Paracetamol vs Ibuprofen: 0.189


In [13]:
from rdkit.Chem import MolFromSmarts

drug_smiles = {'Aspirin':'CC(=O)Oc1ccccc1C(=O)O',
'Paracetamol':'CC(=O)Nc1ccc(O)cc1',
'Ibuprofen':'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
'Metformin':'CN(C)C(=N)NC(N)=N',
'Naproxen':'CC(c1ccc2cc(OC)ccc2c1)C(=O)O'}

cooh = MolFromSmarts('C(=O)O')

for name, smil_string in smiles.items():
  mol = Chem.MolFromSmiles(smil_string)
  if mol is not None:
    if mol.HasSubstructMatch(cooh):
      print(name, 'contains COOH')

Aspirin contains COOH
Ibuprofen contains COOH
